In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

torch.cuda.is_available()

True

In [3]:
from package import create_dataloader, make_collate_fn

sequences = [[1,2,3,4,5,6,7], [10,20,30,40,50], [100,200,300]]

collate_fn = make_collate_fn(min_len=3, max_len=5)
loader = create_dataloader(sequences, batch_size=4, collate_fn=collate_fn)

# for x1_padded, x1_mask, x2_padded, x2_mask in loader:
#     # x1_padded, x2_padded: (batch, max_crop_len) — the positive pairs
#     # x1_mask, x2_mask: (batch, max_crop_len) — True where real tokens exist
#     pass


In [4]:
next(iter(loader))

(tensor([[ 10,  20,  30,  40,  50],
         [100, 200, 300,   0,   0],
         [  3,   4,   5,   6,   7]]),
 tensor([[ True,  True,  True,  True,  True],
         [ True,  True,  True, False, False],
         [ True,  True,  True,  True,  True]]),
 tensor([[ 10,  20,  30,  40],
         [100, 200, 300,   0],
         [  1,   2,   3,   0]]),
 tensor([[ True,  True,  True,  True],
         [ True,  True,  True, False],
         [ True,  True,  True, False]]))

In [5]:
import random

import torch

from package import (
    ContrastiveTransformerEncoder,
    create_dataloader,
    make_collate_fn,
    nt_xent_loss,
)
from package.pad_and_mask import pad_and_mask

# --- Mock data: 100 sequences (vocab 1-50, length 5-20) ---
random.seed(42)
sequences = [
    [random.randint(1, 50) for _ in range(random.randint(5, 20))]
    for _ in range(100)
]

# --- Config ---
VOCAB_SIZE = 51  # 0 reserved for padding
BATCH_SIZE = 16
EPOCHS = 30
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Setup ---
collate_fn = make_collate_fn(min_len=3, max_len=10)
loader = create_dataloader(sequences, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
model = ContrastiveTransformerEncoder(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [6]:
# --- Train ---
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for x1, mask1, x2, mask2 in loader:
        x1, mask1 = x1.to(DEVICE), mask1.to(DEVICE)
        x2, mask2 = x2.to(DEVICE), mask2.to(DEVICE)

        z1 = model(x1, mask1)
        z2 = model(x2, mask2)
        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss / len(loader):.4f}")

# --- Embed ---
model.eval()
padded, mask = pad_and_mask(sequences)
padded, mask = padded.to(DEVICE), mask.to(DEVICE)

with torch.no_grad():
    embeddings = model.encode(padded, mask)

print(f"Embeddings shape: {embeddings.shape}")  # (100, 64)


Epoch 5/30 — Loss: 1.4893
Epoch 10/30 — Loss: 1.2751
Epoch 15/30 — Loss: 1.3718
Epoch 20/30 — Loss: 1.0536
Epoch 25/30 — Loss: 0.8905
Epoch 30/30 — Loss: 0.7114
Embeddings shape: torch.Size([100, 64])


In [7]:
model.eval()
test_seq = [
    [1,2,3,4,5],
    [2,3,4],
    [10,20,30,40],
    [20,30,40,50],
    [20,3,30,4,50]
]

padded, mask = pad_and_mask(test_seq)
padded, mask = padded.to(DEVICE), mask.to(DEVICE)

with torch.no_grad():
    test_embeddings = model.encode(padded, mask)

In [8]:
test_embeddings.shape

torch.Size([5, 64])

In [9]:
test_embeddings

tensor([[-0.0697,  0.6158,  0.3007, -0.6728,  0.1317, -0.3421, -0.1993, -0.9466,
         -0.2705, -0.9390,  0.2555, -0.2644, -0.5680,  0.1048,  0.3225,  0.0939,
          0.5947, -0.1506, -0.3413, -0.1081, -0.2195, -0.5025,  0.2670, -0.1791,
          0.1758, -0.3002, -0.0584,  0.0746, -0.3493,  0.5894, -0.0667, -0.8077,
         -0.3796,  0.3930,  0.4562, -0.3436, -0.7969, -0.4710,  0.8320,  0.5303,
         -1.0616,  1.3546,  0.3383,  0.3214, -0.2352, -0.0907, -0.1696,  0.1653,
          0.2888,  0.5382, -0.1788, -0.1164,  0.1105,  0.3987,  0.1247,  0.2777,
          0.0795, -0.4590,  0.8086, -0.0521,  0.6559,  0.4167,  0.2829, -0.1497],
        [-0.4911,  0.6292,  0.8218, -1.2044, -0.2047, -0.6936, -0.2159, -0.8904,
         -0.2716, -1.2848,  0.8823, -0.4882, -0.2175,  0.4455,  1.0450,  0.9313,
          0.5417,  0.4258, -0.8083, -0.1325,  0.4408, -0.7620,  0.3104, -0.2930,
         -0.4142, -0.6471,  0.3845,  0.1872, -0.3400,  0.8351, -0.0376, -0.2719,
         -0.6831,  0.7340, 

In [10]:
import torch.nn.functional as F

normed = F.normalize(test_embeddings, dim=-1)  # (100, 64)
sim_matrix = normed @ normed.T             # (100, 100)
sim_matrix.shape


torch.Size([5, 5])

In [11]:
sim_matrix

tensor([[1.0000, 0.7478, 0.2528, 0.1329, 0.5110],
        [0.7478, 1.0000, 0.3739, 0.3064, 0.6363],
        [0.2528, 0.3739, 1.0000, 0.7280, 0.6194],
        [0.1329, 0.3064, 0.7280, 1.0000, 0.7180],
        [0.5110, 0.6363, 0.6194, 0.7180, 1.0000]], device='cuda:0')

In [22]:
import umap
import plotly.express as px
import pandas as pd


reducer = umap.UMAP(n_components=3, metric="cosine", random_state=42)
coords = reducer.fit_transform(test_embeddings.cpu().tolist())

df = pd.DataFrame(coords, columns=["x", "y", "z"])

df["sample"] = [f"seq_{i}" for i in range(len(test_seq))]
fig = px.scatter_3d(df, x="x", y="y", z="z", text="sample")
fig.show()

d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


In [21]:
# 2D
reducer_2d = umap.UMAP(n_components=2, metric="cosine", random_state=42)
coords_2d = reducer_2d.fit_transform(test_embeddings.cpu().tolist())

df_2d = pd.DataFrame(coords_2d, columns=["x", "y"])
df_2d["sample"] = [f"seq_{i}" for i in range(len(test_seq))]

fig_2d = px.scatter(df_2d, x="x", y="y", hover_name="sample", title="Embeddings (UMAP 2D)")
fig_2d.show()  # or fig_2d.write_html("embeddings_2d.html")


d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


In [25]:
# --- Save (after training) ---
torch.save(model.state_dict(), "model.pt")

# --- Load (later, for inference) ---
loaded_model = ContrastiveTransformerEncoder(vocab_size=51).to(DEVICE)  # same args as training
loaded_model.load_state_dict(torch.load("model.pt", map_location=DEVICE))
loaded_model.eval()

# Then use model.encode(x, mask) to get embeddings

padded, mask = pad_and_mask(test_seq)
padded, mask = padded.to(DEVICE), mask.to(DEVICE)

with torch.no_grad():
    test_embeddings = loaded_model.encode(padded, mask)

test_embeddings


tensor([[-0.0697,  0.6158,  0.3007, -0.6728,  0.1317, -0.3421, -0.1993, -0.9466,
         -0.2705, -0.9390,  0.2555, -0.2644, -0.5680,  0.1048,  0.3225,  0.0939,
          0.5947, -0.1506, -0.3413, -0.1081, -0.2195, -0.5025,  0.2670, -0.1791,
          0.1758, -0.3002, -0.0584,  0.0746, -0.3493,  0.5894, -0.0667, -0.8077,
         -0.3796,  0.3930,  0.4562, -0.3436, -0.7969, -0.4710,  0.8320,  0.5303,
         -1.0616,  1.3546,  0.3383,  0.3214, -0.2352, -0.0907, -0.1696,  0.1653,
          0.2888,  0.5382, -0.1788, -0.1164,  0.1105,  0.3987,  0.1247,  0.2777,
          0.0795, -0.4590,  0.8086, -0.0521,  0.6559,  0.4167,  0.2829, -0.1497],
        [-0.4911,  0.6292,  0.8218, -1.2044, -0.2047, -0.6936, -0.2159, -0.8904,
         -0.2716, -1.2848,  0.8823, -0.4882, -0.2175,  0.4455,  1.0450,  0.9313,
          0.5417,  0.4258, -0.8083, -0.1325,  0.4408, -0.7620,  0.3104, -0.2930,
         -0.4142, -0.6471,  0.3845,  0.1872, -0.3400,  0.8351, -0.0376, -0.2719,
         -0.6831,  0.7340, 